In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

print("Model training notebook ready!")

Model training notebook ready!


In [2]:
start = time.time()

df = pd.read_pickle("../data/processed/model_data.pkl")

elapsed = time.time() - start

print(f"Loading time: {elapsed:.2f} seconds")
print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

Loading time: 0.56 seconds
Dataset shape: (985989, 47)
Date range: 2013-01-29 00:00:00 to 2015-07-31 00:00:00


In [3]:
# Chronological train-validation split

df = df.sort_values(["Date", "Store"]).reset_index(drop=True)

cutoff_date = df["Date"].max() - pd.Timedelta(weeks=6)

train_df = df[df["Date"] < cutoff_date].copy()
valid_df = df[df["Date"] >= cutoff_date].copy()

print("Cutoff date:", cutoff_date)

print("\nTraining:")
print(train_df["Date"].min(), "to", train_df["Date"].max())
print("Rows:", len(train_df))

print("\nValidation:")
print(valid_df["Date"].min(), "to", valid_df["Date"].max())
print("Rows:", len(valid_df))

Cutoff date: 2015-06-19 00:00:00

Training:
2013-01-29 00:00:00 to 2015-06-18 00:00:00
Rows: 938044

Validation:
2015-06-19 00:00:00 to 2015-07-31 00:00:00
Rows: 47945


In [4]:
# Lag-7 baseline forecast
# Prediction: sales today ≈ sales from the same weekday last week

baseline_pred = valid_df["SalesLag7"]
baseline_actual = valid_df["Sales"]

baseline_mae = mean_absolute_error(
    baseline_actual,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_actual,
        baseline_pred
    )
)

print("Lag-7 Baseline Performance")
print("--------------------------")
print(f"MAE:  {baseline_mae:.2f}")
print(f"RMSE: {baseline_rmse:.2f}")

Lag-7 Baseline Performance
--------------------------
MAE:  1883.03
RMSE: 2614.59


In [5]:
from sklearn.ensemble import RandomForestRegressor

In [7]:
# Remove target, date, and unavailable-at-prediction-time features

excluded_cols = [
    "Sales",
    "Date",
    "Customers"
]

feature_cols = [
    col for col in train_df.columns
    if col not in excluded_cols
]

X_train = train_df[feature_cols]
y_train = train_df["Sales"]

X_valid = valid_df[feature_cols]
y_valid = valid_df["Sales"]

print("Training features:", X_train.shape)
print("Validation features:", X_valid.shape)
print("Number of features:", len(feature_cols))

Training features: (938044, 44)
Validation features: (47945, 44)
Number of features: 44


In [8]:
rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start = time.time()

rf.fit(X_train, y_train)

elapsed = time.time() - start

print(f"\nTraining completed in {elapsed:.2f} seconds")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:   24.8s



Training completed in 38.61 seconds


[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   37.8s finished


In [9]:
# Evaluate Random Forest

start = time.time()

rf_pred = rf.predict(X_valid)

prediction_time = time.time() - start

rf_mae = mean_absolute_error(y_valid, rf_pred)
rf_rmse = np.sqrt(
    mean_squared_error(y_valid, rf_pred)
)

print("Random Forest Performance")
print("-------------------------")
print(f"MAE:  {rf_mae:.2f}")
print(f"RMSE: {rf_rmse:.2f}")
print(f"Prediction time: {prediction_time:.2f} seconds")

print("\nComparison with Lag-7 Baseline")
print("------------------------------")
print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"RF MAE:       {rf_mae:.2f}")
print(f"MAE improvement: {baseline_mae - rf_mae:.2f}")

improvement_pct = (
    (baseline_mae - rf_mae) / baseline_mae
) * 100

print(f"MAE improvement: {improvement_pct:.2f}%")

Random Forest Performance
-------------------------
MAE:  484.35
RMSE: 758.59
Prediction time: 0.21 seconds

Comparison with Lag-7 Baseline
------------------------------
Baseline MAE: 1883.03
RF MAE:       484.35
MAE improvement: 1398.69
MAE improvement: 74.28%


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done  50 out of  50 | elapsed:    0.1s finished


In [10]:
# Analyze feature importance

feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("Top 20 Most Important Features")
print("--------------------------------")
print(feature_importance.head(20))

Top 20 Most Important Features
--------------------------------
               Feature  Importance
0           SalesLag28    0.143034
1                 Open    0.142133
2            DayOfWeek    0.122513
3           SalesLag14    0.099849
4   SalesRollingMean28    0.074677
5            SalesLag1    0.071743
6            SalesLag7    0.071443
7                Promo    0.055690
8     SalesRollingStd7    0.042310
9    SalesRollingMean7    0.038620
10      StateHoliday_0    0.034442
11        SalesChange7    0.026642
12      StateHoliday_a    0.012141
13           DayOfYear    0.009003
14                 Day    0.007568
15          SalesTrend    0.007404
16          WeekOfYear    0.006109
17       SalesMomentum    0.006030
18      StateHoliday_b    0.003892
19         StoreType_b    0.003213


In [11]:
df["SalesRollingMean7"] = (
    df.groupby("Store")["Sales"]
      .transform(
          lambda x: x.shift(1).rolling(7).mean()
      )
)

In [12]:
# Prediction diagnostics

results = valid_df[
    ["Store", "Date", "Sales"]
].copy()

results["PredictedSales"] = rf_pred
results["Error"] = results["Sales"] - results["PredictedSales"]
results["AbsoluteError"] = np.abs(results["Error"])

print("Prediction diagnostics")
print("----------------------")
print(f"Mean Error (Bias): {results['Error'].mean():.2f}")
print(f"Median Absolute Error: {results['AbsoluteError'].median():.2f}")
print(f"90th Percentile Error: {results['AbsoluteError'].quantile(0.90):.2f}")
print(f"Maximum Absolute Error: {results['AbsoluteError'].max():.2f}")

print("\nPrediction sample:")
print(results.head(10))

Prediction diagnostics
----------------------
Mean Error (Bias): -77.65
Median Absolute Error: 340.49
90th Percentile Error: 1119.35
Maximum Absolute Error: 31077.69

Prediction sample:
        Store       Date  Sales  PredictedSales       Error  AbsoluteError
938044      1 2015-06-19   4202     4714.514796 -512.514796     512.514796
938045      2 2015-06-19   4926     5208.912360 -282.912360     282.912360
938046      3 2015-06-19   8074     7928.041968  145.958032     145.958032
938047      4 2015-06-19   9686     9616.025191   69.974809      69.974809
938048      5 2015-06-19   5711     5257.319026  453.680974     453.680974
938049      6 2015-06-19   5464     5284.219385  179.780615     179.780615
938050      7 2015-06-19   9677     9537.123582  139.876418     139.876418
938051      8 2015-06-19   6773     7005.838865 -232.838865     232.838865
938052      9 2015-06-19   7849     8079.834734 -230.834734     230.834734
938053     10 2015-06-19   6191     6243.133594  -52.133594     

In [13]:
# Error analysis by store

store_errors = (
    results.groupby("Store")
    .agg(
        ActualMean=("Sales", "mean"),
        PredictedMean=("PredictedSales", "mean"),
        MAE=("AbsoluteError", "mean"),
        MaxError=("AbsoluteError", "max")
    )
    .sort_values("MAE", ascending=False)
)

print("10 stores with highest MAE")
print("--------------------------")
print(store_errors.head(10))

print("\n10 stores with lowest MAE")
print("-------------------------")
print(store_errors.tail(10))

10 stores with highest MAE
--------------------------
         ActualMean  PredictedMean          MAE      MaxError
Store                                                        
909     8373.697674    7173.794813  4411.922130  31077.694013
876     5989.674419    6271.230470  2553.343159  18197.263220
842    17527.837209   17140.782039  1753.442400   6354.599597
292     3208.511628    3604.173495  1692.541795  20384.456468
756    14618.627907   14526.091440  1490.572022   6589.590611
279     8475.465116    8602.614909  1452.939994   5960.372511
1114   19598.232558   18808.980719  1422.758995   5497.983768
1027   11186.325581   11039.061866  1396.117397   5968.158980
251    16051.627907   16413.734558  1222.613682   5420.128102
335    13444.069767   13702.493531  1153.505924   6367.569371

10 stores with lowest MAE
-------------------------
        ActualMean  PredictedMean         MAE     MaxError
Store                                                     
703    2661.116279    2823.5638

In [14]:
# Relative error analysis by store

store_errors["RelativeMAE_pct"] = (
    store_errors["MAE"]
    / store_errors["ActualMean"].replace(0, np.nan)
) * 100

store_errors = store_errors.sort_values(
    "RelativeMAE_pct",
    ascending=False
)

print("10 stores with highest relative MAE")
print("-----------------------------------")

print(
    store_errors[
        [
            "ActualMean",
            "PredictedMean",
            "MAE",
            "RelativeMAE_pct"
        ]
    ].head(10)
)

print("\nOverall relative MAE summary")
print("----------------------------")

print(
    store_errors["RelativeMAE_pct"].describe()
)

10 stores with highest relative MAE
-----------------------------------
        ActualMean  PredictedMean          MAE  RelativeMAE_pct
Store                                                          
292    3208.511628    3604.173495  1692.541795        52.751618
909    8373.697674    7173.794813  4411.922130        52.687860
876    5989.674419    6271.230470  2553.343159        42.629081
636    5717.418605    5771.324712  1139.936073        19.937950
710    4028.186047    4188.936856   721.289070        17.906052
675    3116.162791    3348.085298   554.197670        17.784619
183    4392.720930    4707.687639   760.012999        17.301645
279    8475.465116    8602.614909  1452.939994        17.142894
534    6602.534884    6845.864621  1092.188650        16.541960
13     4519.790698    4498.238292   684.060832        15.134790

Overall relative MAE summary
----------------------------
count    1115.000000
mean        8.225979
std         2.899071
min         3.756768
25%         6.826

In [15]:
# Error analysis by demand level

results["DemandLevel"] = pd.qcut(
    results["Sales"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

demand_errors = (
    results.groupby(
        "DemandLevel",
        observed=True
    )
    .agg(
        ActualMean=("Sales", "mean"),
        PredictedMean=("PredictedSales", "mean"),
        MAE=("AbsoluteError", "mean"),
        MedianAE=("AbsoluteError", "median"),
        MaxError=("AbsoluteError", "max")
    )
)

demand_errors["RelativeMAE_pct"] = (
    demand_errors["MAE"]
    / demand_errors["ActualMean"].replace(0, np.nan)
) * 100

print("Error by demand level")
print("---------------------")
print(demand_errors)

Error by demand level
---------------------
               ActualMean  PredictedMean         MAE    MedianAE  \
DemandLevel                                                        
Low           1513.736513    1652.698029  184.337271    0.004112   
Medium        5085.336421    5282.770648  448.828014  351.423969   
High          6891.461410    7000.208869  536.120354  426.223049   
Very High    10670.908803   10536.335188  768.300666  573.757529   

                 MaxError  RelativeMAE_pct  
DemandLevel                                 
Low          10646.221087        12.177633  
Medium        7488.388580         8.825926  
High         13842.194178         7.779487  
Very High    31077.694013         7.199955  
